In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql
SELECT * FROM catalog.bronze.visits_raw;

In [0]:
from pyspark.sql.functions import col, lag, try_to_date, coalesce, datediff, when, current_timestamp, to_date
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
bronze_table = 'catalog.bronze.visits_raw'
silver_table = 'catalog.silver.fact_visits'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/fact_visits/checkpoint/"

In [0]:
silver_patients_table = "catalog.silver.dim_patients"
silver_diagnosis_table = "catalog.silver.dim_diagnosis"
silver_hospitals_table = "catalog.silver.dim_hospitals"
silver_physicians_table = "catalog.silver.dim_physicians"

In [0]:
# Load columns needed from each silver table

# For patient data
df_patients = (spark.read.table("catalog.silver.dim_patients")
    .select("patient_id", "first_name", "last_name", "gender", "dob", "city", "assigned_branch", "patient_name_hash")
    .withColumnRenamed("city", "patient_city")
    .withColumnRenamed("assigned_branch", "patient_assigned_branch"))

# For diagnosis-level readmission KPIs
df_diagnosis = (spark.read.table("catalog.silver.dim_diagnosis")
    .select("diagnosis_code", "diagnosis_desc", "diagnosis_category", "readmission_risk_level"))

# For hospital and hospital region KPIs
df_hospitals = (spark.read.table("catalog.silver.dim_hospitals")
    .select("hospital_id", "hospital_name", "city", "network_region_cluster", "hospital_class", "bed_count")
    .withColumnRenamed("city", "hospital_city"))

# For physician KPI
df_physicians = (spark.read.table("catalog.silver.dim_physicians")
    .select("physician_id", "specialization")
    .withColumnRenamed("physician_id", "attending_physician_id"))

In [0]:
df_visits_bronze = (
    spark.readStream.table(bronze_table)
)

In [0]:
window = Window.partitionBy("patient_id").orderBy("admission_date")

# Clean visits stream
df_visits_clean = (
    df_visits_bronze
        .drop(
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date",
            "days_since_last_discharge", # recomputed from clean dates
            "discharge_disposition",     
            "ward_type",
            "admission_type",
            "payment_mode",
            "insurance_claim_no"
        )
        .withColumn("admission_date", coalesce(
            try_to_date(col("admission_date"), "yyyy-MM-dd"),
            try_to_date(col("admission_date"), "MM/dd/yyyy"),
            try_to_date(col("admission_date"), "dd-MM-yyyy"),
            try_to_date(col("admission_date"), "MMMM dd yyyy")))
        .withColumn("discharge_date", coalesce(
            try_to_date(col("discharge_date"), "yyyy-MM-dd"),
            try_to_date(col("discharge_date"), "MM/dd/yyyy"),
            try_to_date(col("discharge_date"), "dd-MM-yyyy"),
            try_to_date(col("discharge_date"), "MMMM dd yyyy")))
        # Recompute LOS from clean dates
        .withColumn("length_of_stay_days",
            when(
            col("discharge_date") >= col("admission_date"),
            datediff(col("discharge_date"), col("admission_date"))
        ).otherwise(None))
        .withColumn("is_30day_readmission",
            when(col("is_30day_readmission") == "Y", True)
            .when(col("is_30day_readmission") == "N", False)
            .otherwise(None))
        )

# Join clean fact visit with dimension tables
df_fact_combined = (
    df_visits_clean
        .join(df_patients, "patient_id", "left")
        .join(df_hospitals, "hospital_id", "left")
        .join(df_diagnosis, "diagnosis_code", "left")
        .join(df_physicians, "attending_physician_id", "left")
        .withColumn("admission_date", to_date("admission_date"))
        .withColumn("discharge_date", to_date("discharge_date"))
        .withColumn("load_timestamp", current_timestamp())
        )

In [0]:
# Merge into silver fact visit
def merge_fact_visits(batch_df, batch_id):
    # Deduplicate on visit_id — keep latest ingested
    batch_df = batch_df.dropDuplicates(["visit_id"])

    # Recompute is_30day_readmission from clean dates
    window = Window.partitionBy("patient_id").orderBy("admission_date")

    batch_df = (batch_df
        .withColumn("prev_discharge_date",
            lag("discharge_date", 1).over(window))
        .withColumn("days_since_last_discharge",
            datediff(col("admission_date"), col("prev_discharge_date")))
        .withColumn("is_30day_readmission",
            when(col("days_since_last_discharge").between(0, 30), True)
            .otherwise(False))
        .drop("prev_discharge_date")    # intermediate calculation column - help compute days_since_last_discharge and is_30day_readmission
        .withColumn("load_timestamp", current_timestamp()))
    

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    fact = DeltaTable.forName(spark, silver_table)

    (fact.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.visit_id = s.visit_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute())

In [0]:
# Run as availableNow increment
(
    df_fact_combined.writeStream
        .foreachBatch(merge_fact_visits)  # for every batch, merge_fact_visits is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.fact_visits;

In [0]:
# Data quality check
from pyspark.sql.functions import col, count, sum as spark_sum, min, max, avg, round

print("FACT VISITS")
df = spark.read.table("catalog.silver.fact_visits")
total = df.count()

print(f"Total rows: {total}")
print("-" * 50)

# Null check
print("\nNull counts:")
for c in ["visit_id", "patient_id", "hospital_id", "diagnosis_code",
          "admission_date", "discharge_date", "is_30day_readmission",
          "length_of_stay_days", "total_cost_php"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n}")

# Duplicates
dupes = total - df.select("visit_id").distinct().count()
print(f"\nDuplicate visit_ids: {'OK' if dupes == 0 else dupes}")

# Readmission split
print("\nReadmission breakdown:")
df.groupBy("is_30day_readmission").count().show()

# LOS and cost ranges
print("LOS range:")
df.select(
    min("length_of_stay_days").alias("min"),
    max("length_of_stay_days").alias("max"),
    round(avg("length_of_stay_days"), 1).alias("avg")
).show()

print("Cost range (PHP):")
df.select(
    min("total_cost_php").alias("min"),
    max("total_cost_php").alias("max"),
    round(avg("total_cost_php"), 0).alias("avg")
).show()

# Readmission rate by risk level
print("Readmission rate by risk level:")
(df.groupBy("readmission_risk_level")
   .agg(
       count("visit_id").alias("visits"),
       spark_sum(col("is_30day_readmission").cast("int")).alias("readmissions")
   )
   .withColumn("rate_%", round(col("readmissions") / col("visits") * 100, 1))
   .orderBy("rate_%", ascending=False)
   .show())